# ViShell — pipeline trên Colab hoặc Kaggle

Notebook mỏng: (mount Drive nếu là Colab) → cài đặt → chọn config → chạy
`python -m vishell pipeline` → hiện `REPORT.md`. Toàn bộ logic nằm trong package
`vishell/` — sửa code ở đó, không sửa trong notebook này (AGENT.md mục 9).

- **Colab**: Runtime > Change runtime type > GPU (T4). Output lưu lên Google Drive.
- **Kaggle**: Settings > Accelerator > GPU T4, và bật **Internet: On** (cần để `git clone`/`pip`).
  Output lưu ở `/kaggle/working` (chỉ giữ lại khi bấm *Save Version*).

In [ ]:
import os

# Kaggle trước: image Kaggle cũng có /content và google.colab, nhưng drive.mount không chạy được ở đó.
if os.path.exists('/kaggle/working'):  # Kaggle
    ROOT = '/kaggle/working/vishell'
    WORK = '/kaggle/working'
elif os.path.exists('/content'):  # Colab
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = '/content/drive/MyDrive/vishell'
    WORK = '/content'
else:
    raise RuntimeError('Không nhận ra Colab hay Kaggle — chạy trên laptop thì dùng `python -m vishell ...` trực tiếp.')
print('ROOT =', ROOT)

In [ ]:
REPO_URL = 'https://github.com/lenhuthuc/GenBashCode.git'  # repo private thì dùng https://<TOKEN>@github.com/lenhuthuc/GenBashCode.git
import subprocess
REPO_DIR = f'{WORK}/TheWicknessHero'
if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:  # runtime cũ: lấy code + pin thư viện mới nhất
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
%cd {REPO_DIR}/vishell
!pip install -q -e .[train,data]

In [ ]:
SMOKE = True  # True: cấu hình smoke (nhanh, bắt lỗi sớm) — False: chạy thật
CONFIG = 'configs/smoke.yaml' if SMOKE else 'configs/experiment_t4.yaml'  # thí nghiệm A/B/C; colab_t4.yaml = chạy đủ mọi hệ
# smoke.yaml mặc định dùng fixture nhỏ nên bỏ qua bước dịch; trỏ tới đường dẫn không tồn tại để pipeline
# tự dịch 50 câu qua vLLM (thử luôn bước dịch). Cấu hình thật (SMOKE=False) đã tự dịch toàn bộ.
EXTRA = '--set data.nl2bash_dir=data/nl2bash_vi' if SMOKE else ''
print('Config:', CONFIG, EXTRA)

In [ ]:
# Một lệnh chạy từ dịch NL2Bash -> SFT -> GRPO -> đánh giá -> báo cáo; bước nào xong rồi tự bỏ qua.
# smoke.yaml tắt bước GPU cho laptop (train.enabled=false) — ở đây có GPU nên bật lại.
!python -m vishell pipeline --config {CONFIG} --set paths.root={ROOT!r} --set train.enabled=true {EXTRA}

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open('REPORT.md', encoding='utf-8').read()))